# Notebook 04 — Benchmarks and Evaluation

**What you'll learn:** How to define **benchmark questions** for your Genie space — questions with known SQL answers that let you measure accuracy over time.

**Why this matters:** When you change instructions, add examples, or Databricks updates the model, benchmarks tell you whether Genie got better or worse. Think of them as **unit tests for your Genie space**.

**What this notebook does:**
1. Defines 10 benchmark questions with ground-truth SQL.
2. Verifies each ground-truth query runs correctly against your data.
3. Pushes the benchmarks to the Genie Benchmarks tab via the `serialized_space` API.
4. Guides you to **run benchmarks in the Genie UI** (the only accurate evaluator).
5. Shows how to **fix failures by updating instructions** programmatically.

**Before you start:** Run notebooks **02** (data) and **03** (spaces).

**Compute:** Serverless.


## Compute

Use **Serverless**. Catalog and schema come from **widgets** only.


## Tips for writing good benchmarks

Benchmarks are the **unit tests of your Genie space**. They let you measure whether Genie answers correctly — and catch regressions when you change instructions or models.

- Ask for **one clear number or small result** per question (easy to score automatically).
- Compute expected answers in **SQL from the same tables** Genie uses — no manual guesswork.
- Mix **counts, averages, and filters** (e.g. OEE, downtime, defects).
- Use questions **real users would ask**; update benchmarks when the schema changes.
- In notebook 07 (compare), allow a small tolerance for rounding on ratio queries.
- The **curated SQL examples** in notebook 03 teach Genie *style*; benchmarks here track *correctness* over time.
- If the API step errors, you can add the same questions under **Genie Evaluation / Benchmarks** in the UI.

## Load config and print Genie space links

Reads the space IDs saved by notebook 03 and prints clickable links. After this notebook pushes benchmarks, open the **primary space** link to try them out.

In [ ]:
%run ./00_workshop_config

In [ ]:
import re
import json
import requests
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

host = w.config.host.rstrip("/")
headers = {**w.config.authenticate(), "Content-Type": "application/json"}

def genie_ui_room_url(space_id):
    m = re.search(r"adb-(\d+)\.", host)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host}/genie/rooms/{space_id}{q}"

_cfg = spark.table(full_table("workshop_config")).toPandas().set_index("key")["value"].to_dict()
PRIMARY_SPACE_ID = _cfg.get("genie_space_id", "")
BLANK_SPACE_ID = _cfg.get("genie_space_id_blank", "")
NO_EVALS_SPACE_ID = _cfg.get("genie_space_id_configured_no_evals", "")

if not PRIMARY_SPACE_ID:
    raise RuntimeError("genie_space_id not found in workshop_config. Run notebook 03 first.")

print("Your Genie spaces (click to open):")
print(f"  Configured: {genie_ui_room_url(PRIMARY_SPACE_ID)}")
if NO_EVALS_SPACE_ID:
    print(f"  No Examples: {genie_ui_room_url(NO_EVALS_SPACE_ID)}")
if BLANK_SPACE_ID:
    print(f"  Blank (no instructions):  {genie_ui_room_url(BLANK_SPACE_ID)}")
print()
print("Benchmarks will be pushed to the PRIMARY space.")
print("After this notebook completes, open the primary space link above")
print("and try asking one of the benchmark questions to verify it works.")

## Define benchmark suite

10 benchmark questions across two tiers:

**Core (Q1–Q7):** Counts, sums, filters, and joins that any well-configured space should get right.

**Pattern-teaching (Q8–Q10):** Harder questions that teach Genie the SQL patterns it needs (state joins, status filters, shift aggregation). These are intentionally **different** from the 4 evaluation questions in notebook 07 — Genie must generalize from these patterns, not memorize answers.


In [ ]:
benchmarks_v1 = [
    # --- Core benchmarks (Q1-Q7) ---
    {
        "q": "How many distinct production lines had at least one defect_detected event in calendar year 2024?",
        "sql": f"""SELECT CAST(COUNT(DISTINCT production_line_id) AS BIGINT) AS answer FROM {fqn}.production_events WHERE event_type = 'defect_detected' AND YEAR(CAST(event_date AS DATE)) = 2024""",
        "gt": f"SELECT CAST(COUNT(DISTINCT production_line_id) AS BIGINT) FROM {fqn}.production_events WHERE event_type = 'defect_detected' AND YEAR(CAST(event_date AS DATE)) = 2024",
    },
    {
        "q": "What is the sum of downtime_minutes from quality_metrics_daily for all rows in 2024 where the plant is in Michigan (join to plants by state)?",
        "sql": f"""SELECT SUM(q.downtime_minutes) AS total_downtime_minutes FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Michigan' AND q.date >= '2024-01-01' AND q.date <= '2024-12-31'""",
        "gt": f"SELECT SUM(q.downtime_minutes) FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Michigan' AND q.date >= '2024-01-01' AND q.date <= '2024-12-31'",
    },
    {
        "q": "How many production lines had 500 or more unit_produced events in calendar year 2024?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM ( SELECT production_line_id FROM {fqn}.production_events WHERE event_type = 'unit_produced' AND YEAR(CAST(event_date AS DATE)) = 2024 GROUP BY production_line_id HAVING COUNT(*) >= 500 ) t""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM ( SELECT production_line_id FROM {fqn}.production_events WHERE event_type = 'unit_produced' AND YEAR(CAST(event_date AS DATE)) = 2024 GROUP BY production_line_id HAVING COUNT(*) >= 500 ) t",
    },
    {
        "q": "How many operators work at plants located in Michigan (use operators.plant_id joined to plants)?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM {fqn}.operators o JOIN {fqn}.plants p ON o.plant_id = p.plant_id WHERE p.state = 'Michigan'""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.operators o JOIN {fqn}.plants p ON o.plant_id = p.plant_id WHERE p.state = 'Michigan'",
    },
    {
        "q": "For calendar year 2024 quality_metrics_daily only, what is the overall defect rate in percent rounded to a whole number: 100 times the sum of defects_found divided by the sum of units_produced?",
        "sql": f"""SELECT ROUND(100.0 * SUM(defects_found) / SUM(units_produced)) AS defect_rate_percent FROM {fqn}.quality_metrics_daily WHERE date >= '2024-01-01' AND date <= '2024-12-31'""",
        "gt": f"SELECT ROUND(100.0 * SUM(defects_found) / SUM(units_produced)) FROM {fqn}.quality_metrics_daily WHERE date >= '2024-01-01' AND date <= '2024-12-31'",
    },
    {
        "q": "How many equipment_feedback rows are for production lines at the EV Battery Innovation Center (join feedback to production_lines and match that plant)?",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM {fqn}.equipment_feedback f JOIN {fqn}.production_lines pl ON f.production_line_id = pl.line_id JOIN {fqn}.plants p ON pl.plant_id = p.plant_id WHERE p.plant_name = 'EV Battery Innovation Center'""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.equipment_feedback f JOIN {fqn}.production_lines pl ON f.production_line_id = pl.line_id JOIN {fqn}.plants p ON pl.plant_id = p.plant_id WHERE p.plant_name = 'EV Battery Innovation Center'",
    },
    {
        "q": "For plant P003 only and calendar year 2024, what is average OEE as a whole percent rounded to an integer (average oee_score times 100, then round)?",
        "sql": f"""SELECT ROUND(AVG(q.oee_score) * 100) AS average_oee_percent FROM {fqn}.quality_metrics_daily q WHERE q.plant_id = 'P003' AND q.date >= '2024-01-01' AND q.date <= '2024-12-31'""",
        "gt": f"SELECT ROUND(AVG(q.oee_score) * 100) FROM {fqn}.quality_metrics_daily q WHERE q.plant_id = 'P003' AND q.date >= '2024-01-01' AND q.date <= '2024-12-31'",
    },
    # --- Pattern-teaching benchmarks (Q8-Q10) ---
    # These teach patterns needed by the 4 north-star eval questions but use DIFFERENT filters/states.
    {
        "q": "What is the scrap rate for Texas plants for all of 2024? Return scrap_count divided by units_produced as a percentage from quality_metrics_daily, rounded to 2 decimal places.",
        "sql": f"""SELECT CAST(ROUND(100.0 * SUM(q.scrap_count) / NULLIF(SUM(q.units_produced), 0), 2) AS DOUBLE) AS answer FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Texas' AND YEAR(CAST(q.date AS DATE)) = 2024""",
        "gt": f"SELECT CAST(ROUND(100.0 * SUM(q.scrap_count) / NULLIF(SUM(q.units_produced), 0), 2) AS DOUBLE) FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Texas' AND YEAR(CAST(q.date AS DATE)) = 2024",
    },
    {
        "q": "How many production lines currently have status Active? Return the count.",
        "sql": f"""SELECT CAST(COUNT(*) AS BIGINT) AS answer FROM {fqn}.production_lines WHERE status = 'Active'""",
        "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM {fqn}.production_lines WHERE status = 'Active'",
    },
    {
        "q": "How many inspection_passed events did the busiest shift have in Q1 2024? Join production_events to operators by operator_id to get the shift, group by shift, and return only the highest count.",
        "sql": f"""SELECT CAST(MAX(cnt) AS BIGINT) AS answer FROM ( SELECT o.shift, COUNT(*) AS cnt FROM {fqn}.production_events pe JOIN {fqn}.operators o ON pe.operator_id = o.operator_id WHERE pe.event_type = 'inspection_passed' AND CAST(pe.event_date AS DATE) >= DATE '2024-01-01' AND CAST(pe.event_date AS DATE) < DATE '2024-04-01' GROUP BY o.shift ) t""",
        "gt": f"SELECT CAST(MAX(cnt) AS BIGINT) FROM ( SELECT o.shift, COUNT(*) AS cnt FROM {fqn}.production_events pe JOIN {fqn}.operators o ON pe.operator_id = o.operator_id WHERE pe.event_type = 'inspection_passed' AND CAST(pe.event_date AS DATE) >= DATE '2024-01-01' AND CAST(pe.event_date AS DATE) < DATE '2024-04-01' GROUP BY o.shift ) t",
    },
]
print(f"{len(benchmarks_v1)} benchmarks defined")


## Verify ground truth in Spark

Run each `gt` query once. Fix definitions before pushing to Genie if any fail.


In [ ]:
for i, b in enumerate(benchmarks_v1, 1):
    try:
        v = spark.sql(b["gt"]).collect()[0][0]
        print(f"OK Q{i}: {v}")
    except Exception as e:
        print(f"FAIL Q{i}: {e}")


## Push benchmarks to the Genie Benchmarks tab

Pushes benchmarks to the Genie space using the GA `serialized_space` API (`GET` with `include_serialized_space=true`, then `PATCH`). This reads the current space configuration, replaces only the `benchmarks.questions` section, and writes the full blob back — preserving all other settings (instructions, curated examples, sample questions, tables).

These appear in the **Benchmark** tab in the Genie UI (not the SQL Queries section). Replaces existing benchmarks so the notebook is idempotent on re-run.

> **Note:** You can also add benchmarks manually in the Genie UI (Benchmark tab → Add question). The programmatic approach here is convenient for bulk loading and reproducibility.


In [ ]:
import uuid

def push_benchmarks(benchmarks_list, space_id, label=""):
    """Push benchmarks via the GA serialized_space API (GET-merge-PATCH).

    Reads the current space config with include_serialized_space=true,
    replaces only benchmarks.questions, and PATCHes the full blob back.
    All other config (instructions, curated examples, tables) is preserved.
    """
    # 1. READ current space config
    current = requests.get(
        f"{host}/api/2.0/genie/spaces/{space_id}",
        headers=headers,
        params={"include_serialized_space": "true"},
    ).json()
    config = json.loads(current.get("serialized_space", "{}"))
    if not config:
        print(f"ERROR: Could not read serialized_space for {space_id}. Is the space ID correct?")
        return

    prev_count = len(config.get("benchmarks", {}).get("questions", []))

    # 2. MODIFY — build benchmark questions in the serialized_space format
    benchmark_questions = []
    for b in benchmarks_list:
        benchmark_questions.append({
            "id": uuid.uuid4().hex,
            "question": [b["q"]],
            "answer": [{"format": "SQL", "content": [b["sql"]]}],
        })

    config.setdefault("benchmarks", {})["questions"] = benchmark_questions
    config["version"] = 2

    # 3. WRITE back
    r = requests.patch(
        f"{host}/api/2.0/genie/spaces/{space_id}",
        headers=headers,
        json={"serialized_space": json.dumps(config)},
    )
    if r.status_code == 200:
        print(f"{label}: {len(benchmarks_list)} benchmarks pushed (replaced {prev_count} existing)")
    else:
        print(f"PATCH failed: {r.status_code} {r.text[:300]}")

push_benchmarks(benchmarks_v1, PRIMARY_SPACE_ID, "Primary space benchmarks")
print(f"\nOpen the Benchmarks tab to verify: {genie_ui_room_url(PRIMARY_SPACE_ID)}")
print("Click 'Benchmark' in the top nav → you should see all 10 questions listed.")


## Run benchmarks in the Genie UI

There is **no API** to trigger the Genie benchmark evaluator programmatically. The benchmark evaluator is a **UI-only feature** that:

1. Asks Genie each benchmark question
2. Runs **both** Genie's generated SQL and your ground-truth SQL
3. Compares the **full result sets** (not just a single number)
4. Shows a SQL diff analysis for any mismatches

**To run benchmarks:**

1. Open the Genie space link printed below
2. Click the **Benchmark** tab in the top nav bar
3. Click **Start new run** (top right)
4. Wait for all questions to complete
5. Review pass/fail results — click any failed question to see the failure analysis


In [ ]:
print("Open the Benchmarks tab to evaluate:")
print(f"  {genie_ui_room_url(PRIMARY_SPACE_ID)}/benchmarks")
print()
print("Click 'Start new run' and review the results.")
print("Note any failures — we'll fix them in the next steps.")


## Fix failures: Two tools the UI gives you

After a benchmark run, you have **two ways** to fix failures. Use both.

### Tool 1: Accept knowledge snippets (primary fix)

When a benchmark fails, the UI extracts **knowledge snippets** — patterns Genie can learn from the failure. Click **Review proposed fixes** on any failed benchmark to see them.

You'll see snippets in categories like:

| Category | Example | What it teaches Genie |
|---|---|---|
| **FILTER** | `plants.state = 'Michigan'` means "plant is in Michigan" | Use exact match instead of ILIKE |
| **JOIN** | `quality_metrics_daily JOINS plants ON q.plant_id = p.plant_id` | Correct join path between tables |
| **MEASURE** | `ROUND(AVG(oee_score) * 100)` means "average OEE as a whole percent" | How to calculate OEE |
| **MEASURE** | `ROUND(100.0 * SUM(defects_found) / SUM(units_produced))` means "defect rate" | How to calculate defect rate |
| **FILTER** | `YEAR(CAST(date AS DATE)) = 2024` means "calendar year 2024" | Preferred date filtering style |

**These are suggestions, not guaranteed fixes.** Use your judgment and domain expertise to review each snippet before accepting. Ask yourself: "Is this the SQL pattern I want Genie to use for all future questions like this?" Accept the ones that match your data conventions, reject any that would teach Genie the wrong pattern for your use case.

### Tool 2: Update ground truth SQL (secondary fix)

Some failures are **style differences**, not logic errors — Genie and your ground truth produce the same result but with different SQL. For example:

| What Genie writes | What ground truth has | Same result? |
|---|---|---|
| `ROUND(AVG(oee_score) * 100)` | `CAST(ROUND(AVG(oee_score) * 100, 0) AS BIGINT)` | Yes |
| `date >= '2024-01-01' AND date <= '2024-12-31'` | `YEAR(CAST(date AS DATE)) = 2024` | Yes |
| `try_divide(a, NULLIF(b, 0))` | `a / b` | Yes (on clean data) |

For these, click **Update ground truth** on the failed benchmark and replace your SQL with Genie's version. This is not cheating — it's aligning the test with Genie's natural style so the benchmark evaluator (which compares full result sets) marks it as passing. Again, use your SME judgment — only update ground truth if you confirm the two queries return the same result for your data.


## Advanced: Update instructions programmatically

If accepting knowledge snippets and updating ground truth is not enough — for example, Genie keeps
making the same mistake across multiple questions — you can **add rules to the space instructions** via the API.

The cell below reads the current space configuration (using `include_serialized_space=true`), appends clarifying rules to the `text_instructions`, and PATCHes it back. All other settings (curated examples, benchmarks, tables) are preserved.

Edit `ADDITIONAL_RULES` based on the patterns you see failing, then run the cell and re-run benchmarks in the UI.


In [ ]:
# Additional instructions to address common benchmark failure patterns.
# Edit this list based on the failures you see in the benchmark UI.
ADDITIONAL_RULES = [
    "When filtering by state or plant_name, use exact match (= 'Michigan') not ILIKE or partial match.",
    "For year-based filters, both YEAR(CAST(date AS DATE)) = 2024 and date >= '2024-01-01' AND date <= '2024-12-31' are acceptable.",
    "For integer counts, cast the final result with CAST(... AS BIGINT). For rates/percentages, use CAST(... AS DOUBLE).",
]

# 1. READ current space config (include_serialized_space=true returns the full blob)
current = requests.get(
    f'{host}/api/2.0/genie/spaces/{PRIMARY_SPACE_ID}',
    headers=headers,
    params={'include_serialized_space': 'true'},
).json()
config = json.loads(current.get('serialized_space', '{}'))
if not config:
    raise RuntimeError("Could not read serialized_space. Check that PRIMARY_SPACE_ID is correct.")

# 2. MODIFY — append rules to existing text_instructions (content is a list of strings)
existing_ti = config.get('instructions', {}).get('text_instructions', [])
if existing_ti and existing_ti[0].get('content'):
    existing_content = existing_ti[0]['content']
    existing_id = existing_ti[0].get('id', uuid.uuid4().hex)
else:
    existing_content = []
    existing_id = uuid.uuid4().hex

already_has_rules = any('Additional Rules:' in line for line in existing_content)
if not already_has_rules:
    new_rule_lines = ['', 'Additional Rules:'] + [f'- {r}' for r in ADDITIONAL_RULES]
    updated_content = existing_content + new_rule_lines
    print(f'Appending {len(ADDITIONAL_RULES)} rules to instructions.')
else:
    updated_content = existing_content
    print('Additional Rules already present — skipping to avoid duplicates.')

config.setdefault('instructions', {})['text_instructions'] = [{'id': existing_id, 'content': updated_content}]
config['version'] = 2

# 3. WRITE back — full replace, but we read first so everything is preserved
r = requests.patch(
    f'{host}/api/2.0/genie/spaces/{PRIMARY_SPACE_ID}',
    headers=headers,
    json={'serialized_space': json.dumps(config)},
)
if r.status_code == 200:
    print(f'Instructions updated. Total lines: {len(updated_content)}')
else:
    print(f'PATCH failed: {r.status_code} {r.text[:300]}')

print(f'\nRe-run benchmarks in the UI to verify improvement:')
print(f'  {genie_ui_room_url(PRIMARY_SPACE_ID)}')


## Iterate until 100%

The improvement loop:

1. **Run benchmarks** in the UI (Benchmark tab, Start new run)
2. **Review failures** — click each failed question to see the failure analysis
3. **Accept knowledge snippets** — click Review proposed fixes, accept correct patterns
4. **Update ground truth** — if the SQL is equivalent but styled differently, update it
5. **Programmatic fix** (optional) — if a pattern keeps failing, add a rule in the cell above
6. **Re-run** — click Start new run and check if the fixes worked

**Targets:**
- **100%** on synthetic data (like this workshop) — achievable because the data is clean and controlled
- **85%+** on real customer data — some ambiguity is expected with real-world schemas

**Next:** Notebook 05 — Explore with Genie (interactive Q&A).


## Smoke test: 4 hard questions

Once your benchmarks are green in the UI, run these 4 advanced questions against the Configured space.
These are **not** part of the benchmarks — they test whether Genie can **generalize** from your instructions
and curated examples to answer questions it has never seen before.

| # | Question | What makes it hard |
|---|---|---|
| Q1 | Scrap rate for Michigan, last 5 days of 2024 | State join + date range + ratio calculation |
| Q2 | Plants with a Maintenance-status line | Static status field (no date filter needed) |
| Q3 | Lines with >5% defect rate in 2024 | CASE/HAVING subquery across event types |
| Q4 | Best-quality shift in Dec 2024 | Events-to-operators join + MIN aggregation |

These same 4 questions are used in **notebook 07** (A/B comparison) to prove that curated examples matter.


In [ ]:
import time

def _extract_number(text):
    if text is None:
        return None
    import re as _re
    nums = _re.findall(r"-?\d+(?:\.\d+)?", str(text).replace(",", ""))
    return float(nums[0]) if nums else None

# NOTE: Genie Conversation API (Public Preview as of April 2026)
def ask_genie(question, space_id, timeout=120):
    try:
        s = requests.post(f"{host}/api/2.0/genie/spaces/{space_id}/start-conversation",
            headers=headers, json={"content": question})
        if s.status_code != 200: return None, f"HTTP {s.status_code}"
        d = s.json()
        cid, mid = d.get("conversation_id"), d.get("message_id")
        for _ in range(timeout // 4):
            time.sleep(4)
            p = requests.get(f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}", headers=headers)
            if p.status_code != 200: continue
            msg = p.json()
            if msg.get("status") == "COMPLETED":
                for att in msg.get("attachments", []):
                    aid = att.get("attachment_id") or att.get("id", "")
                    if not att.get("query") or not aid: continue
                    qr = requests.get(f"{host}/api/2.0/genie/spaces/{space_id}/conversations/{cid}/messages/{mid}/query-result/{aid}", headers=headers)
                    if qr.status_code == 200:
                        arr = qr.json().get("statement_response",{}).get("result",{}).get("data_array",[])
                        if arr and arr[0]: return _extract_number(arr[0][0]), None
                return None, "No numeric value"
            elif msg.get("status") in ("FAILED","CANCELLED"): return None, msg.get("status")
        return None, "Timeout"
    except Exception as e: return None, str(e)[:80]

TOLERANCE_PCT = 5.0

hard_questions = [
    {"q": "What is the scrap rate for Michigan plants for the last 5 days of 2024? Return the total scrap_count divided by total units_produced as a percentage from quality_metrics_daily.",
     "gt": f"SELECT CAST(ROUND(100.0 * SUM(q.scrap_count) / NULLIF(SUM(q.units_produced), 0), 2) AS DOUBLE) FROM {fqn}.quality_metrics_daily q JOIN {fqn}.plants p ON q.plant_id = p.plant_id WHERE p.state = 'Michigan' AND CAST(q.date AS DATE) >= DATE '2024-12-27' AND CAST(q.date AS DATE) <= DATE '2024-12-31'",
     "short": "Scrap rate Michigan last 5 days"},
    {"q": "How many distinct plants have at least one production line with status Maintenance? Return the count of distinct plants.",
     "gt": f"SELECT CAST(COUNT(DISTINCT p.plant_id) AS BIGINT) FROM {fqn}.production_lines pl JOIN {fqn}.plants p ON pl.plant_id = p.plant_id WHERE pl.status = 'Maintenance'",
     "short": "Plants with Maintenance lines"},
    {"q": "How many production lines have a high historical defect rate \u2014 meaning more than 5% of their production_events in 2024 were defect_detected out of total unit_produced events?",
     "gt": f"SELECT CAST(COUNT(*) AS BIGINT) FROM ( SELECT production_line_id, COUNT(CASE WHEN event_type = 'defect_detected' THEN 1 END) AS defects, COUNT(CASE WHEN event_type = 'unit_produced' THEN 1 END) AS produced FROM {fqn}.production_events WHERE YEAR(CAST(event_date AS DATE)) = 2024 GROUP BY production_line_id HAVING produced > 0 AND (100.0 * defects / produced) > 5.0 ) t",
     "short": "Lines with >5% defect rate"},
    {"q": "How many defect_detected events did the best-quality shift have in December 2024? Best quality = fewest defects. Join production_events to operators by operator_id to get the shift, group by shift, and return only the lowest defect count.",
     "gt": f"SELECT CAST(MIN(defect_count) AS BIGINT) FROM ( SELECT o.shift, COUNT(*) AS defect_count FROM {fqn}.production_events pe JOIN {fqn}.operators o ON pe.operator_id = o.operator_id WHERE pe.event_type = 'defect_detected' AND CAST(pe.event_date AS DATE) >= DATE '2024-12-01' AND CAST(pe.event_date AS DATE) <= DATE '2024-12-31' GROUP BY o.shift ) t",
     "short": "Best-quality shift Dec 2024"},
]

print("Smoke test: 4 hard questions against the Configured space")
print(f"Space: {genie_ui_room_url(PRIMARY_SPACE_ID)}")
print()
passes = 0
for i, hq in enumerate(hard_questions, 1):
    gt_val = float(spark.sql(hq["gt"]).collect()[0][0])
    genie_val, err = ask_genie(hq["q"], PRIMARY_SPACE_ID)
    if genie_val is not None and gt_val != 0:
        pct_diff = abs(genie_val - gt_val) / abs(gt_val) * 100
        status = "PASS" if pct_diff <= TOLERANCE_PCT else "FAIL"
    elif genie_val is not None and gt_val == 0:
        status = "PASS" if genie_val == 0 else "FAIL"
    else: status = "FAIL"
    if status == "PASS": passes += 1
    print(f"  Q{i} ({hq['short']}): {status} (GT={gt_val}, Genie={genie_val}{f', err={err}' if err else ''})")

rate = passes / len(hard_questions) * 100
print(f"\nSmoke test: {passes}/{len(hard_questions)} ({rate:.0f}%)")
if rate == 100: print("All 4 hard questions passed. Ready for notebook 05.")
else: print("Some questions failed. Review instructions and curated examples in the template.")
